In [28]:
import numpy as np
import jax.numpy as jnp

In [22]:
p_ti = np.arange(12).reshape((3, 4))
theta_ti = np.arange(12).reshape((3, 4))
print(f"{p_ti=}")
print(f"{theta_ti=}")

p_ti=array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])
theta_ti=array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])


In [9]:
mult = p_ti * theta_ti
print(f"{mult=}")

mult=array([[  0,   1,   4,   9],
       [ 16,  25,  36,  49],
       [ 64,  81, 100, 121]])


In [16]:
n_t = np.arange(3) + 1
print(f"{n_t=}")

n_t=array([1, 2, 3])


In [7]:
n_t = np.full(
            shape=(3, ),
            fill_value=1.0,
        )

In [8]:
n_t

array([1., 1., 1.])

In [17]:
mult / n_t[:, None]

array([[ 0.        ,  1.        ,  4.        ,  9.        ],
       [ 8.        , 12.5       , 18.        , 24.5       ],
       [21.33333333, 27.        , 33.33333333, 40.33333333]])

In [ ]:
result = (X == np.arange(X.shape[0])[:, None]).astype(int)

In [3]:
X = np.arange(12)
X %= 4
X

array([0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3])

In [13]:
%timeit (X == jnp.arange(5)[:, None]).astype(int)

633 μs ± 19.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
%timeit (X == np.arange(5)[:, None]).astype(int)

2.1 μs ± 5.19 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [15]:
B = [5, 3, 1, 3, 6]
P = np.arange(15).reshape((3, 5))
print(f"{B=}")
print(f"{P=}")

B=[5, 3, 1, 3, 6]
P=array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14]])


In [16]:
T, I = P.shape
W = 7
# Предполложение: токены в B находятся в диапазоне [0, W-1]
N = np.zeros((T, W), dtype=P.dtype)

# Индексы строк (T, 1) + токены (I,) -> broadcast в (T, I)
rows = np.arange(T)[:, None]
np.add.at(N, (rows, B), P)


In [21]:
n_topics = 3
n_t_tilda = np.full(
            shape=(n_topics, ),
            fill_value=1.0,
        )
print(f"{n_t_tilda=}")

n_t_tilda=array([1., 1., 1.])


In [25]:
np.sum(p_ti, axis=1)

array([ 6, 22, 38])

In [26]:
n_t_tilda += np.sum(p_ti, axis=1)

In [27]:
print(f"{n_t_tilda=}")

n_t_tilda=array([ 7., 23., 39.])


In [31]:
from cartm.preprocessing import DatasetPreprocessor
with open('./data/test_data.txt') as f:
    data = f.readlines()
print(f'Total number of documents in corpus: {len(data)}')
print(f'Total number of words in corpus: {sum([len(doc.split(" ")) for doc in data])}')
preprocessor = DatasetPreprocessor(stopwords=[], min_word_len=0)
tokenized_data, document_bounds = preprocessor.fit_transform(data)
print(f'Total number of document boundaries in preprocessed corpus: {len(document_bounds)}')
print(f'Total number of tokenized words in preprocessed corpus: {len(tokenized_data)}')

Total number of documents in corpus: 4
Total number of words in corpus: 34
Total number of document boundaries in preprocessed corpus: 5
Total number of tokenized words in preprocessed corpus: 34


In [37]:
print(f"{tokenized_data=}")
print(f"{document_bounds=}")
batch_data = [tokenized_data[:document_bounds[2]], tokenized_data[document_bounds[2]:]]
print(f"{batch_data=}")
batch_document_bounds = [document_bounds[:3], document_bounds[2:]]
print(f"{batch_document_bounds=}")

tokenized_data=Array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
       10, 14, 17, 20,  6,  4, 11, 23,  3, 16, 19, 28, 12, 23,  5, 23, 15],      dtype=int32)
document_bounds=Array([ 0, 10, 19, 27, 34], dtype=int32)
batch_data=[Array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
       10, 14], dtype=int32), Array([17, 20,  6,  4, 11, 23,  3, 16, 19, 28, 12, 23,  5, 23, 15], dtype=int32)]
batch_document_bounds=[Array([ 0, 10, 19], dtype=int32), Array([19, 27, 34], dtype=int32)]


In [164]:
#from cartm.preprocessing import BatchLoader

#batch_loader = BatchLoader(data=tokenized_data, doc_bounds=document_bounds, batch_size=20)
#[(batch, doc_bounds) for batch, doc_bounds in batch_loader]

[(Array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
         10, 14, 17], dtype=int32),
  Array([ 0, 10, 19, 20], dtype=int32)),
 (Array([20,  6,  4, 11, 23,  3, 16, 19, 28, 12, 23,  5, 23, 15], dtype=int32),
  Array([ 0,  7, 14], dtype=int32))]

In [137]:
from numpy.typing import NDArray

def _norm_numpy(x: NDArray, eps: float = 1e-6) -> NDArray:
    x = np.maximum(x, np.zeros_like(x))
    norm = x.sum(axis=0)
    return np.divide(x, norm, out=np.zeros_like(x), where=norm != 0)

def bidir_ema(
                X: NDArray[np.float64], 
                indices: NDArray[np.int64], 
                gamma=0.6, 
                beta=0.5
             ) -> NDArray[np.float64]:
    
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    H, I = X.shape
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    arange = np.arange(I)
    alpha_pow = alpha ** arange
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in, idx_in):
        W = X_in * alpha_inv_pow
        global_cum = np.cumsum(W, axis=1)
        
        resets = idx_in[1:-1]
        mask = np.zeros(I, dtype=int)
        if len(resets) > 0:
            mask[resets] = resets
            
        last_reset = np.maximum.accumulate(mask)
        correction = np.zeros((H, I))
        valid = last_reset > 0
        if np.any(valid):
            correction[:, valid] = global_cum[:, last_reset[valid] - 1]
        
        seg_cum = global_cum - correction
        
        lengths = np.diff(idx_in)
        starts = idx_in[:-1]
        X_starts = np.repeat(X_in[:, starts], lengths, axis=1)
        alpha_inv_starts = np.repeat(alpha_inv_pow[starts], lengths)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    right_ema = calc_right_ema(X, indices)
    
    X_rev = X[:, ::-1]
    rev_indices = I - indices[::-1]
    left_ema_rev = calc_right_ema(X_rev, rev_indices)
    left_ema = left_ema_rev[:, ::-1]
    
    return beta * right_ema + (1.0 - beta) * left_ema

In [67]:
seed = 42
_eps = 1e-3
n_topics = 3
vocab_size = len(preprocessor.vocabulary)
gamma = 0.6

In [51]:
np.random.seed(seed=seed)

phi = _norm_numpy(np.random.uniform(
    size=(n_topics, vocab_size)),
    eps=_eps
)

n_t = np.full(
    shape=(n_topics, ),
    fill_value=1.0,
)

n_w = np.zeros(vocab_size)

print(f"{phi.shape=}")
print(f"{np.sum(phi, axis=0)=}")
print(f"{n_t=}")
print(f"{n_w=}")

phi.shape=(3, 29)
np.sum(phi, axis=0)=array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
n_t=array([1., 1., 1.])
n_w=array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [144]:
#for it in range(max_iter): #// шаг 2 - начало цикла проходов по всей коллекции
#// шаг 3 инициализация
n_tw: NDArray = np.zeros_like(phi)
N_tw: NDArray = np.zeros_like(phi)
n_t_tilda: NDArray = np.zeros_like(n_t)
#// конец шаг 3 инициализация

print(f"{n_tw.shape=}")
print(f"{N_tw.shape=}")
print(f"{n_t_tilda=}")

n_tw.shape=(3, 29)
N_tw.shape=(3, 29)
n_t_tilda=array([0., 0., 0.])


In [145]:
#// цикл для всех батчей шаги 4-11
#for batch in data:
batch = batch_data[0]
ctx_bounds = batch_document_bounds[0]

print(f"{batch.shape=}")
print(f"{ctx_bounds=}")

batch.shape=(19,)
ctx_bounds=Array([ 0, 10, 19], dtype=int32)


In [146]:
#// шаг 5 
p_ti: NDArray = phi[:, batch] #// шаг 5 (I, T) строки (распределение по темам) для токенов по порядку позиций в батче 
print(f"{p_ti.shape=}")
print(f"{batch=}")
print(f"{p_ti=}")
print(f"{phi[:, 9]=}")
print(f"{phi[:, 14]=}")

print(f"{p_ti[:, (1, 4)]=}")
print(f"{phi[:, 8]=}")

p_ti.shape=(3, 19)
batch=Array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
       10, 14], dtype=int32)
p_ti=array([[0.31602021, 0.745955  , 0.11772564, 0.37823745, 0.745955  ,
        0.56508663, 0.54526961, 0.23646155, 0.47556409, 0.26668645,
        0.10091857, 0.58864877, 0.13265026, 0.51427315, 0.83559531,
        0.47556409, 0.52419313, 0.02943229, 0.15059794],
       [0.31048429, 0.10088692, 0.67876709, 0.11445572, 0.10088692,
        0.0824148 , 0.42011781, 0.29749866, 0.06094491, 0.5425885 ,
        0.81341249, 0.10844956, 0.39412165, 0.16228485, 0.0811112 ,
        0.06094491, 0.30441516, 0.83789175, 0.84600142],
       [0.3734955 , 0.15315808, 0.20350727, 0.50730684, 0.15315808,
        0.35249858, 0.03461258, 0.46603979, 0.463491  , 0.19072506,
        0.08566894, 0.30290167, 0.47322809, 0.32344199, 0.08329348,
        0.463491  , 0.17139172, 0.13267596, 0.00340064]])
phi[:, 9]=array([0.31602021, 0.31048429, 0.3734955 ])
phi[:, 14]=array([0.15059794

In [147]:
#// шаг 7 
theta_ti = bidir_ema(
    X=np.asarray(p_ti), 
    indices=np.asarray(ctx_bounds), 
    gamma=gamma, 
    beta=0.5
)

print(f"{theta_ti.shape=}")
print(f"{theta_ti=}")

theta_ti.shape=(3, 19)
theta_ti=array([[0.36362335, 0.56400849, 0.28319252, 0.4179191 , 0.62052088,
        0.54565619, 0.5016195 , 0.33160041, 0.41157948, 0.29957834,
        0.17620022, 0.43544174, 0.27367832, 0.49012658, 0.66675271,
        0.48701447, 0.44017966, 0.15471577, 0.16678495],
       [0.29704579, 0.21400883, 0.46902485, 0.19261539, 0.1517724 ,
        0.15525535, 0.33114734, 0.28886925, 0.20464432, 0.46520806,
        0.68765093, 0.28751971, 0.34574212, 0.2051703 , 0.14839431,
        0.17047533, 0.37022469, 0.71620454, 0.79505583],
       [0.33933086, 0.22198268, 0.24778263, 0.38946552, 0.22770671,
        0.29908846, 0.16723316, 0.37953033, 0.3837762 , 0.2352136 ,
        0.13614886, 0.27703855, 0.38057956, 0.30470312, 0.18485298,
        0.3425102 , 0.18959565, 0.12907969, 0.03815922]])


In [148]:
print(f"{p_ti[:, 0]=}")
print(f"{theta_ti[:, 0]=}")
print(f"{n_t=}")

#// шаг 8 
print(f"{(p_ti * theta_ti / n_t[:, None])[:, 0]=}")
p_ti = _norm_numpy(p_ti * theta_ti / n_t[:, None])

print(f"{p_ti[:, 0]=}")

p_ti[:, 0]=array([0.31602021, 0.31048429, 0.3734955 ])
theta_ti[:, 0]=array([0.36362335, 0.29704579, 0.33933086])
n_t=array([1., 1., 1.])
(p_ti * theta_ti / n_t[:, None])[:, 0]=array([0.11491233, 0.09222805, 0.12673855])
p_ti_new[:, 0]=array([0.34417365, 0.27623202, 0.37959433])


In [149]:
#// шаг 9 - расчет q_wi
wi_equal_w = (batch == np.arange(phi.shape[1])[:, None]).astype(float)
q_wi = bidir_ema(
    wi_equal_w, 
    indices=np.asarray(ctx_bounds), 
    gamma=gamma, 
    beta=0.5
)

print(f"{batch=}")
print(f"{ctx_bounds=}")
print(f"{wi_equal_w[8, :]=}")
print(f"{q_wi[8, :]=}")

batch=Array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
       10, 14], dtype=int32)
ctx_bounds=Array([ 0, 10, 19], dtype=int32)
wi_equal_w[8, :]=Array([0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0.], dtype=float32)
q_wi[8, :]=array([0.12768   , 0.6192    , 0.168     , 0.168     , 0.6192    ,
       0.12768   , 0.051072  , 0.0204288 , 0.00817152, 0.00326861,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        ])


In [150]:
print(f"{q_wi.shape=}")
print(f"{p_ti.shape=}")
print(f"{theta_ti.shape=}")


q_wi.shape=(29, 19)
p_ti.shape=(3, 19)
theta_ti.shape=(3, 19)


In [151]:
'''
q_wi - матрица WxI, в ячейке влияние токена w на позицию i (аналог внимания)
p_ti - матрица TxI, в ячейке вероятность темы t в позиции i (посчитанное с учетом контекста C_i и токена w_i)
theta_ti - матрица TxI, в ячейке вероятность темы t в позиции i (свертка контекстов на основе скользящей средней по токенам, находящимся в контексте C_i)

Допустим у нас W=100 слов в словаре, размер батча I = 20 токенов, хотим найти T=5 тем.

Как рассчитать N_tw по этой формуле?
Ведь N_tw - матрица размером TxW, где в ячейках частота встречаемости пары токен w и тема t 
'''

#// шаг 10 
#N_tw: NDArray = np.zeros_like(phi)
N_tw += (p_ti / theta_ti) @ q_wi.T

print(f"{p_ti[0, :]=}")
print(f"{theta_ti[0, :]=}")

print(f"{(p_ti / theta_ti)[1, :]=}")

print(f"{q_wi[1, :]=}")
print(f"{N_tw[1, :]=}")

p_ti[0, :]=array([0.31602021, 0.745955  , 0.11772564, 0.37823745, 0.745955  ,
       0.56508663, 0.54526961, 0.23646155, 0.47556409, 0.26668645,
       0.10091857, 0.58864877, 0.13265026, 0.51427315, 0.83559531,
       0.47556409, 0.52419313, 0.02943229, 0.15059794])
theta_ti[0, :]=array([0.36362335, 0.56400849, 0.28319252, 0.4179191 , 0.62052088,
       0.54565619, 0.5016195 , 0.33160041, 0.41157948, 0.29957834,
       0.17620022, 0.43544174, 0.27367832, 0.49012658, 0.66675271,
       0.48701447, 0.44017966, 0.15471577, 0.16678495])
(p_ti / theta_ti)[1, :]=array([1.0452405 , 0.47141476, 1.44718791, 0.59421897, 0.66472507,
       0.53083385, 1.26867337, 1.02987306, 0.29780894, 1.16633513,
       1.18288576, 0.37719001, 1.13992951, 0.79097636, 0.5465924 ,
       0.35749986, 0.82224434, 1.16990566, 1.064078  ])
q_wi[1, :]=array([0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.0012288, 0.003072 ,
       0.00768  , 0.01

In [152]:
ptr = [1.04387599, 0.52778057, 1.42198174, 0.6443587 , 0.71451656,
       0.57940341, 1.25102192, 1.03113321, 0.36268604, 1.15926959,
       1.17466639, 0.44102962, 1.13303829, 0.81200366, 0.60677095,
       0.41889948, 0.84634676, 1.16315893, 1.06153781]
qr = [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.0012288, 0.003072 ,
       0.00768  , 0.0192   , 0.048    , 0.12     , 0.6      , 0.12     ,
       0.048    ]
sum([pt * q for pt, q in zip(ptr, qr)])

0.804824363071872

In [153]:
#// шаг 11.1 n_tw_i := n_tw_i + p_ti
#n_tw: NDArray = np.zeros_like(phi)
rows = np.arange(n_topics)[:, None]
np.add.at(n_tw, (rows, batch), p_ti)

print(f"{batch=}")
print(f"{p_ti[0, (1, 4)]=}")
print(f"{sum(p_ti[0, (1, 4)])=}")
print(f"{n_tw[0, 8]=}")

batch=Array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
       10, 14], dtype=int32)
p_ti[0, (1, 4)]=array([0.745955, 0.745955])
sum(p_ti[0, (1, 4)])=np.float64(1.491910001554498)
n_tw[0, 8]=np.float64(1.491910001554498)


In [154]:
#// шаг 11.2 
#n_t_tilda: NDArray = np.zeros_like(n_t)
n_t_tilda += np.sum(p_ti, axis=1)

print(f"{p_ti[0, :]=}")
print(f"{sum(p_ti[0, :])=}")
print(f"{n_t_tilda[0]=}")

p_ti[0, :]=array([0.31602021, 0.745955  , 0.11772564, 0.37823745, 0.745955  ,
       0.56508663, 0.54526961, 0.23646155, 0.47556409, 0.26668645,
       0.10091857, 0.58864877, 0.13265026, 0.51427315, 0.83559531,
       0.47556409, 0.52419313, 0.02943229, 0.15059794])
sum(p_ti[0, :])=np.float64(7.744835135599719)
n_t_tilda[0]=np.float64(7.744835135599719)


In [155]:
#// шаг ~11.3
#n_w = np.zeros(vocab_size)
n_w += np.bincount(batch, minlength=phi.shape[1])

print(f"{batch=}")
print(f"{n_w=}")
print(f"{n_w[8]=}")

batch=Array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
       10, 14], dtype=int32)
n_w=array([1., 1., 1., 0., 0., 0., 0., 1., 2., 1., 1., 0., 0., 2., 1., 0., 0.,
       0., 1., 0., 0., 1., 1., 0., 1., 1., 1., 1., 1.])
n_w[8]=np.float64(2.0)


In [156]:
print(f"{N_tw[0, :]=}")
print(f"{n_w=}")

#[(t, n) for t, n in zip(N_tw[0, :], n_w)]

N_tw[0, :]=array([1.1163972 , 0.9867454 , 1.02426103, 0.        , 0.        ,
       0.        , 0.        , 0.98796673, 2.11083255, 1.04600215,
       0.4478213 , 0.        , 0.        , 1.94854225, 0.91062232,
       0.        , 0.        , 0.        , 0.82275315, 0.        ,
       0.        , 0.82485172, 0.69618491, 0.        , 1.05724115,
       1.01801266, 0.64844172, 1.02356637, 0.89804297])
n_w=array([1., 1., 1., 0., 0., 0., 0., 1., 2., 1., 1., 0., 0., 2., 1., 0., 0.,
       0., 1., 0., 0., 1., 1., 0., 1., 1., 1., 1., 1.])


In [157]:
#// конец цикл для всех батчей шаги 4-11

phi_new = _norm_numpy(n_tw + np.divide(n_tw * N_tw, n_w, out=np.zeros_like(n_tw), where=n_w != 0))
n_t = n_t_tilda

print(f"{n_w=}")
print(f"{phi_new=}")

n_w=array([1., 1., 1., 0., 0., 0., 0., 1., 2., 1., 1., 0., 0., 2., 1., 0., 0.,
       0., 1., 0., 0., 1., 1., 0., 1., 1., 1., 1., 1.])
phi_new=array([[0.86638856, 0.54609833, 0.61014072, 0.        , 0.        ,
        0.        , 0.        , 0.52326557, 0.77719994, 0.30897475,
        0.02205311, 0.        , 0.        , 0.47246906, 0.13484173,
        0.        , 0.        , 0.        , 0.22161934, 0.        ,
        0.        , 0.08797515, 0.11397651, 0.        , 0.26272033,
        0.54685621, 0.09718286, 0.57678092, 0.36366702],
       [0.06385487, 0.28421424, 0.08666967, 0.        , 0.        ,
        0.        , 0.        , 0.14643537, 0.08539673, 0.30863377,
        0.85276597, 0.        , 0.        , 0.04668071, 0.86291548,
        0.        , 0.        , 0.        , 0.2922067 , 0.        ,
        0.        , 0.83545828, 0.38436834, 0.        , 0.55303633,
        0.42641957, 0.71459779, 0.06974495, 0.09930376],
       [0.06975657, 0.16968743, 0.30318961, 0.        , 0.     

In [158]:
diff_norm = np.linalg.norm(phi_new - phi)

In [159]:
diff_norm

np.float64(0.11441386553846457)

In [143]:
phi = phi_new

In [171]:
%load_ext autoreload
%autoreload 2

In [1]:
from matplotlib.pylab import ma
from sys import path as syspath
from os import path as ospath
import jax.numpy as jnp

from sklearn.datasets import fetch_20newsgroups
from cartm.my_attentive_model import MyAttentiveTopicModel

from cartm.preprocessing import DatasetPreprocessor
from time import time

In [3]:
data = fetch_20newsgroups(data_home='./data/', subset='all').data
data = data[:400]
#with open('./data/test_data.txt') as f:
#    data = f.readlines()

preprocessor = DatasetPreprocessor(
    stopwords=set(), 
    min_word_len=0
    )

batch_data = preprocessor.fit_transform_batch(data, max_batch_size=1000)
#print(batch_data)

In [5]:
len(batch_data)

112

In [10]:
sum(b[1].shape[0] - 1 for b in batch_data)

400

In [11]:
attentive_topic_model = MyAttentiveTopicModel(
    vocab_size=len(preprocessor.vocabulary),
    ctx_len=3,
    n_topics=3
)

attentive_topic_model.fit(
    batch_data_with_doc_bounds=batch_data,
    seed=42,
)

/home/kn/project_artm/topic-modelling-attention/src/cartm/my_attentive_model.py:49: RuntimeWarning: divide by zero encountered in divide
  alpha_inv_pow = 1.0 / alpha_pow
/home/kn/project_artm/topic-modelling-attention/src/cartm/my_attentive_model.py:49: RuntimeWarning: overflow encountered in divide
  alpha_inv_pow = 1.0 / alpha_pow
/home/kn/project_artm/topic-modelling-attention/src/cartm/my_attentive_model.py:74: RuntimeWarning: invalid value encountered in multiply
  term2 = gamma * alpha_pow * seg_cum
/home/kn/project_artm/topic-modelling-attention/src/cartm/my_attentive_model.py:30: RuntimeWarning: invalid value encountered in divide
  return np.divide(x, norm, out=np.zeros_like(x), where=norm != 0)
/home/kn/project_artm/topic-modelling-attention/src/cartm/my_attentive_model.py:52: RuntimeWarning: invalid value encountered in multiply
  W = X_in * alpha_inv_pow
/home/kn/project_artm/topic-modelling-attention/src/cartm/my_attentive_model.py:66: RuntimeWarning: invalid value encoun

it=0
diff_norm=np.float64(nan)
it=1
diff_norm=np.float64(nan)


KeyboardInterrupt: 

KeyboardInterrupt: 